# WMDP Full Dataset Scale Test: No-Corrupt Tiny GPT-2

This notebook is the next gate after the smoke test. It clones the project from GitHub, loads a real tiny causal LM through the project `HFModel` wrapper, runs WMDP bio/chem/cyber on the full local dataset with no corruption, and writes summary/prediction artifacts.

Purpose: verify full-dataset scale, batching, output serialization, and Kaggle runtime behavior. The model is `sshleifer/tiny-gpt2`, so scores are not meaningful for research conclusions.


In [1]:
from pathlib import Path
import importlib
import json
import os
import subprocess
import sys
import time

REPO_URL = 'https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git'
REPO_DIR_NAME = 'PoRT_LLM_Unlearning-Experiment'
IS_KAGGLE = Path('/kaggle/working').exists()

def has_project_layout(path):
    path = Path(path)
    return (path / 'llm-unlearn-eco' / 'eco').exists() and (path / 'dataset').exists()

def clone_or_use_project():
    if IS_KAGGLE:
        target = Path('/kaggle/working') / REPO_DIR_NAME
        if has_project_layout(target):
            print(f'Using existing cloned repository: {target}')
            subprocess.check_call(['git', '-C', str(target), 'pull', '--ff-only'])
            return target.resolve()
        if target.exists():
            raise RuntimeError(f'{target} exists but does not look like this repo.')
        print(f'Cloning {REPO_URL} into {target}')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
        return target.resolve()

    local_root = Path.cwd().resolve()
    if has_project_layout(local_root):
        return local_root
    target = local_root / REPO_DIR_NAME
    if has_project_layout(target):
        return target.resolve()
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(target)])
    return target.resolve()

PROJECT_ROOT = clone_or_use_project()
ECO_ROOT = PROJECT_ROOT / 'llm-unlearn-eco'
os.environ['PORT_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(ECO_ROOT) not in sys.path:
    sys.path.insert(0, str(ECO_ROOT))

commit_sha = subprocess.check_output(['git', '-C', str(PROJECT_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print(f'Project root: {PROJECT_ROOT}')
print(f'Commit: {commit_sha}')


Cloning https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git into /kaggle/working/PoRT_LLM_Unlearning-Experiment


Cloning into '/kaggle/working/PoRT_LLM_Unlearning-Experiment'...


Project root: /kaggle/working/PoRT_LLM_Unlearning-Experiment
Commit: 7f0fd8a8bbe5b4aee939952dabd4199c483a5cc6


In [2]:
required_packages = {
    'datasets': 'datasets>=2.10.1',
    'pandas': 'pandas',
    'pyarrow': 'pyarrow>=10',
    'tabulate': 'tabulate',
    'tqdm': 'tqdm',
    'transformers': 'transformers>=4.38.0',
    'accelerate': 'accelerate>=0.26.0',
}

missing_packages = []
for module_name, package_spec in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ModuleNotFoundError:
        missing_packages.append(package_spec)

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('Required packages are already available.')


Required packages are already available.


In [3]:
import gc
import platform
import pandas as pd
import torch

from eco.dataset import WMDPBio, WMDPChem, WMDPCyber
from eco.evaluator import ChoiceByTopLogit
from eco.inference import EvaluationEngine
from eco.model import HFModel
from eco.paths import MODEL_CONFIG_DIR

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        print(f'GPU {idx}: {props.name}, VRAM={props.total_memory / 1024**3:.2f} GiB')


Python: 3.12.13
Torch: 2.10.0+cpu
CUDA available: False


In [4]:
MODEL_NAME = os.environ.get('PORT_REAL_MODEL_NAME', 'tiny-gpt2')
BATCH_SIZE = int(os.environ.get('PORT_WMDP_BATCH_SIZE', '32'))
RUN_NAME = f'wmdp_full_no_corrupt_{MODEL_NAME}'
RUN_DIR = (Path('/kaggle/working') if IS_KAGGLE else PROJECT_ROOT / 'results') / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)

run_config = {
    'repo_url': REPO_URL,
    'commit_sha': commit_sha,
    'model_name': MODEL_NAME,
    'batch_size': BATCH_SIZE,
    'corrupt_method': None,
    'sample_size': None,
    'datasets': ['wmdp-bio', 'wmdp-chem', 'wmdp-cyber'],
    'run_dir': str(RUN_DIR),
}
(RUN_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
print(json.dumps(run_config, indent=2))


{
  "repo_url": "https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git",
  "commit_sha": "7f0fd8a8bbe5b4aee939952dabd4199c483a5cc6",
  "model_name": "tiny-gpt2",
  "batch_size": 32,
  "corrupt_method": null,
  "sample_size": null,
  "datasets": [
    "wmdp-bio",
    "wmdp-chem",
    "wmdp-cyber"
  ],
  "run_dir": "/kaggle/working/wmdp_full_no_corrupt_tiny-gpt2"
}


## Load Real Model

This uses the same `HFModel` wrapper that full experiments use. For this scale test the default config points to `sshleifer/tiny-gpt2`.


In [5]:
start_model = time.perf_counter()
model = HFModel(model_name=MODEL_NAME, config_path=str(MODEL_CONFIG_DIR))
if model.tokenizer.pad_token_id is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = 'right'
model_load_seconds = time.perf_counter() - start_model
print(f'Model loaded in {model_load_seconds:.2f}s')


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/29 [00:00<?, ?it/s]

Number of parameters: 203228


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.51M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Model loaded in 3.64s


## Run Full WMDP Dataset


In [6]:
choice_labels = ['A', 'B', 'C', 'D']
all_summaries = []
all_predictions = []
subject_timings = {}

for data_module in [WMDPBio(), WMDPChem(), WMDPCyber()]:
    print(f'\nRunning {data_module.name} full test...')
    subject_start = time.perf_counter()
    engine = EvaluationEngine(
        model=model,
        tokenizer=model.tokenizer,
        data_module=data_module,
        subset_names=['test'],
        evaluator=ChoiceByTopLogit(),
        batch_size=BATCH_SIZE,
    )
    engine.inference()
    summary_stats, outputs = engine.summary()
    elapsed = time.perf_counter() - subject_start
    subject_timings[data_module.name] = elapsed
    all_summaries.extend(summary_stats)

    result_name, batches = list(outputs[0].items())[0]
    row_idx = 0
    for batch in batches:
        correct = batch['correct']
        predicted = batch['predicted']
        for c, p in zip(correct, predicted):
            all_predictions.append({
                'dataset': data_module.name,
                'row_index': row_idx,
                'correct_index': int(c),
                'predicted_index': int(p),
                'correct_label': choice_labels[int(c)] if 0 <= int(c) < len(choice_labels) else str(c),
                'predicted_label': choice_labels[int(p)] if 0 <= int(p) < len(choice_labels) else str(p),
                'is_correct': int(c) == int(p),
            })
            row_idx += 1
    print(f'{data_module.name} completed in {elapsed:.2f}s with {row_idx} rows.')

predictions_df = pd.DataFrame(all_predictions)
summary_by_dataset = predictions_df.groupby('dataset')['is_correct'].agg(['count', 'mean']).reset_index()
summary_by_dataset = summary_by_dataset.rename(columns={'mean': 'accuracy'})

print(summary_by_dataset)



Running wmdp-bio full test...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1273 [00:00<?, ? examples/s]

Map:   0%|          | 0/1273 [00:00<?, ? examples/s]

Evaluating choice_by_top_logit of wmdp-bio on test: 100%|██████████| 40/40 [00:29<00:00,  1.38it/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1987 [00:00<?, ? examples/s]

Map:   0%|          | 0/1987 [00:00<?, ? examples/s]

Evaluating choice_by_top_logit of wmdp-cyber on test: 100%|██████████| 63/63 [05:12<00:00,  4.96s/it]

{'wmdp-cyber_test_choice_by_top_logit': 0.26119778560644186}
wmdp-cyber completed in 312.70s with 1987 rows.
      dataset  count  accuracy
0    wmdp-bio   1273  0.250589
1   wmdp-chem    408  0.247549
2  wmdp-cyber   1987  0.261198


In [7]:
summary_payload = {
    'run_config': run_config,
    'model_load_seconds': model_load_seconds,
    'subject_timings_seconds': subject_timings,
    'engine_summaries': all_summaries,
    'summary_by_dataset': summary_by_dataset.to_dict(orient='records'),
    'total_rows': int(len(predictions_df)),
    'overall_accuracy': float(predictions_df['is_correct'].mean()) if len(predictions_df) else None,
}

summary_path = RUN_DIR / 'summary.json'
predictions_path = RUN_DIR / 'predictions.csv'
summary_by_dataset_path = RUN_DIR / 'summary_by_dataset.csv'

summary_path.write_text(json.dumps(summary_payload, indent=2, default=str), encoding='utf-8')
predictions_df.to_csv(predictions_path, index=False)
summary_by_dataset.to_csv(summary_by_dataset_path, index=False)

print(f'Wrote {summary_path}')
print(f'Wrote {predictions_path}')
print(f'Wrote {summary_by_dataset_path}')
print(json.dumps(summary_payload, indent=2, default=str)[:4000])


Wrote /kaggle/working/wmdp_full_no_corrupt_tiny-gpt2/summary.json
Wrote /kaggle/working/wmdp_full_no_corrupt_tiny-gpt2/predictions.csv
Wrote /kaggle/working/wmdp_full_no_corrupt_tiny-gpt2/summary_by_dataset.csv
{
  "run_config": {
    "repo_url": "https://github.com/toanthangO20/PoRT_LLM_Unlearning-Experiment.git",
    "commit_sha": "7f0fd8a8bbe5b4aee939952dabd4199c483a5cc6",
    "model_name": "tiny-gpt2",
    "batch_size": 32,
    "corrupt_method": null,
    "sample_size": null,
    "datasets": [
      "wmdp-bio",
      "wmdp-chem",
      "wmdp-cyber"
    ],
    "run_dir": "/kaggle/working/wmdp_full_no_corrupt_tiny-gpt2"
  },
  "model_load_seconds": 3.638450596000098,
  "subject_timings_seconds": {
    "wmdp-bio": 29.414376013000037,
    "wmdp-chem": 7.4438505179999765,
    "wmdp-cyber": 312.69602824699996
  },
  "engine_summaries": [
    {
      "wmdp-bio_test_choice_by_top_logit": 0.25058915946582877
    },
    {
      "wmdp-chem_test_choice_by_top_logit": 0.24754901960784315
    },

In [8]:
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('WMDP FULL DATASET TINY-GPT2 SCALE TEST COMPLETED')


WMDP FULL DATASET TINY-GPT2 SCALE TEST COMPLETED
